# 뉴스 데이터 xml 수집하기 - sbs news

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

news_rss = requests.get('https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07')
news_rss_soup = BeautifulSoup(news_rss.content, 'xml')
link_list = news_rss_soup.select('item > link')
print("기사 개수:", len(link_list))

기사 개수: 29


In [ ]:
title_list = news_rss_soup.select('item > title')
title_list = [title.text for title in title_list]
print(title_list)

In [2]:
news_df = pd.DataFrame(columns=['title', 'url', 'body'], index=['No'])
news_count = 0

for link in link_list:
    
    retry = 0 
    while (True) :
        news_response = requests.get(link.text, timeout=10)
        if news_response.status_code == 200 :
            break
        else :
            retry += 1
            if retry > 3 : break
            
    if news_response.status_code != 200 :
        print("### 기사를 가져오는 데 실패했습니다.!!!")
        print("URL :", link.text)
        continue
    
    print('-'*100)
    news_soup = BeautifulSoup(news_response.content, 'html.parser')
    mews_title = news_soup.select_one('div.w_article_title > #news-title').text   
    print("제목 :", mews_title)
    
    news_content = news_soup.select_one("div.text_area[itemprop='articleBody']")
    if news_content is not None :
        news_content = news_content.text.strip()
    else :
        news_content = news_soup.select_one("div.text_area").text.strip()
        print("URL :", link.text)
        
    print(f"본문 :\n{news_content.strip()[:100]}")
    
    news_count += 1
    news_df.loc[news_count] = [mews_title, link.text, news_content[:300]]
    
    
print('*'*100)

news_df.to_csv("news.csv", encoding="utf-8-sig", index=True)
print("Save complete")

----------------------------------------------------------------------------------------------------
제목 : 미 연방법원 "이민자 무연고 제3국 추방은 위법…취소해야"
본문 :
▲ 27일 미니애폴리스 연방법원 광장에서 이민세관단속국(ICE)에 반대하는 시위가 열리고 있다.

 트럼프 미국 행정부가 이민자들을 아무런 연고가 없는 제3국으로 추방하는 최근 정
----------------------------------------------------------------------------------------------------
제목 : "사무실에 누가 있다" 한밤 침입자…반전 정체
본문 :
미국 오하이오주의 한 방송국에서 뜻밖의 불청객이 발견됐습니다. 
  
 쓰레기통을 뒤적거리는 모습, 정체는 바로 라쿤이었는데요. 
  
 직원들은 요즘 안 그래도 사무실 안에 누군
----------------------------------------------------------------------------------------------------
제목 : '갤럭시 S26' 시리즈 공개…AI가 먼저 알아서 제안한다
본문 :
<앵커> 
  
 삼성이 새 갤럭시 스마트폰 시리즈를 발표했습니다. AI가 전화나 메시지를 확인하고, 이용자에게 뭘 하면 좋을지 제안을 하기도 하고, AI 기능이 대폭 강화됐습니다
----------------------------------------------------------------------------------------------------
제목 : 3억 8천만 명 본 '최초의 유튜브'…박물관 걸렸다
본문 :
이제 유튜브 영상도 박물관에 전시되는 시대인가요. 
  
 최근 영국 한 유명 박물관의 소장품 소식이 눈길을 끌고 있습니다. 
  
 2005년 4월 유튜브 공동 창립자 자베드 카
-------------------------------

In [ ]:
# print(news_response.url)
# print(news_response.status_code)
# print(news_content_soup.prettify()[:2000])  # 앞부분만 확인

# 뉴스 컨텐츠 클린징

In [ ]:
news_data_1 = []
for link in link_list:
    news_response = requests.get(link.text)
    news_content_soup = BeautifulSoup(news_response.content, 'html.parser')

    news_content = news_content_soup.select_one("div[itemprop=articleBody]")

    if news_content is not None:
          news_data_1.append(news_content.text.strip())
    else:
        news_data_1.append("본문 없음")  # 또는 None, ""
news_data_1_df = pd.DataFrame(data={'title': title_list, 'content': news_data_1})

In [ ]:
news_data_1_df.head()

In [ ]:
missing_count = sum(1 for content in news_data_1 if content == "본문 없음")
print(f"본문이 없는 기사 개수: {missing_count}개")
print(f"전체 기사 개수: {len(news_data_1)}개")

In [ ]:
import pandas as pd

news_df = pd.DataFrame({
    'title': title_list,
    'content': news_data
})
news_df['is_missing'] = news_df['content'] == "본문 없음"
missing_count = news_df['is_missing'].sum()
total_count = len(news_df)

print(f"본문 없음: {missing_count}개 / 전체: {total_count}개 ({missing_count/total_count:.1%})")